# Essence Wars: Agent Evaluation

This notebook demonstrates how to evaluate and benchmark trained agents against various baselines.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/essence-wars/blob/main/python/notebooks/03_evaluation.ipynb)

## Setup

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install essence-wars[train]

import numpy as np
import torch
from essence_wars import PyGame
from essence_wars._core import STATE_TENSOR_SIZE, ACTION_SPACE_SIZE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Evaluation Functions

In [ ]:
def evaluate_vs_bot(agent_fn, bot_type: str, num_games: int = 100, 
                    deck1: str = 'artificer_tokens', deck2: str = 'broodmother_pack'):
    """Evaluate an agent against a built-in bot.
    
    Args:
        agent_fn: Function that takes (game) and returns action
        bot_type: 'random', 'greedy', 'mcts_50', 'mcts_100'
        num_games: Number of games to play
        deck1: Agent's deck
        deck2: Opponent's deck
        
    Returns:
        dict with win_rate, wins, losses, draws
    """
    wins, losses, draws = 0, 0, 0
    
    for seed in range(num_games):
        game = PyGame(deck1=deck1, deck2=deck2)
        game.reset(seed=seed)
        
        while not game.is_done():
            if game.current_player() == 0:
                # Agent's turn
                action = agent_fn(game)
            else:
                # Opponent's turn
                if bot_type == 'random':
                    action = game.random_action()
                elif bot_type == 'greedy':
                    action = game.greedy_action()
                elif bot_type == 'mcts_50':
                    action = game.mcts_action(50)
                elif bot_type == 'mcts_100':
                    action = game.mcts_action(100)
                else:
                    raise ValueError(f"Unknown bot type: {bot_type}")
            
            game.step(action)
        
        winner = game.winner()
        if winner == 0:
            wins += 1
        elif winner == 1:
            losses += 1
        else:
            draws += 1
    
    return {
        'win_rate': wins / num_games,
        'wins': wins,
        'losses': losses,
        'draws': draws,
        'games': num_games,
    }

print("Evaluation functions defined")

## Evaluate Built-in Bots Against Each Other

First, let's establish baseline performance levels.

In [ ]:
# Random vs Greedy
def random_agent(game):
    return game.random_action()

def greedy_agent(game):
    return game.greedy_action()

print("Baseline comparisons (50 games each):")
print("-" * 50)

# Random vs Greedy
result = evaluate_vs_bot(random_agent, 'greedy', num_games=50)
print(f"Random vs Greedy: {result['win_rate']*100:.1f}% ({result['wins']}/{result['games']})")

# Greedy vs Random
result = evaluate_vs_bot(greedy_agent, 'random', num_games=50)
print(f"Greedy vs Random: {result['win_rate']*100:.1f}% ({result['wins']}/{result['games']})")

# Greedy vs Greedy (deck matchup)
result = evaluate_vs_bot(greedy_agent, 'greedy', num_games=50)
print(f"Greedy vs Greedy: {result['win_rate']*100:.1f}% ({result['wins']}/{result['games']})")

## Load a Trained Model

Load a checkpoint and evaluate against baselines.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class PPONetwork(nn.Module):
    """Simple PPO network (copy from training notebook)."""
    
    def __init__(self, obs_dim: int = STATE_TENSOR_SIZE, 
                 action_dim: int = ACTION_SPACE_SIZE,
                 hidden_dim: int = 256):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.policy = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
        )
        self.value = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )
    
    def forward(self, obs: torch.Tensor, mask: torch.Tensor):
        features = self.shared(obs)
        logits = self.policy(features)
        logits = logits.masked_fill(~mask.bool(), float('-inf'))
        value = self.value(features)
        return logits, value.squeeze(-1)

def load_model(checkpoint_path: str):
    """Load a trained model from checkpoint."""
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config = checkpoint.get('config', {})
    hidden_dim = config.get('hidden_dim', 256)
    
    network = PPONetwork(hidden_dim=hidden_dim).to(device)
    network.load_state_dict(checkpoint['network_state_dict'])
    network.eval()
    return network

# Example: Load model if exists
# network = load_model('ppo_quickstart.pt')
print("Model loading function defined")

In [ ]:
def make_neural_agent(network):
    """Create an agent function from a neural network."""
    def agent_fn(game):
        obs = torch.from_numpy(game.observe()).float().unsqueeze(0).to(device)
        mask = torch.from_numpy(game.action_mask()).float().unsqueeze(0).to(device)
        with torch.no_grad():
            logits, _ = network(obs, mask)
            action = logits.argmax(dim=-1).item()
        return action
    return agent_fn

# Example evaluation (uncomment if you have a trained model)
# network = load_model('ppo_quickstart.pt')
# neural_agent = make_neural_agent(network)
# 
# print("Evaluating trained model:")
# print("-" * 50)
# for bot in ['random', 'greedy', 'mcts_50']:
#     result = evaluate_vs_bot(neural_agent, bot, num_games=50)
#     print(f"vs {bot:10s}: {result['win_rate']*100:5.1f}% ({result['wins']}/{result['games']})")

## Using the Benchmark Suite

The `EssenceWarsBenchmark` class provides standardized evaluation with Elo rating calculation.

In [ ]:
from essence_wars.benchmark import EssenceWarsBenchmark, GreedyAgent, RandomAgent

# Create benchmark suite
benchmark = EssenceWarsBenchmark(
    games_per_opponent=20,  # Games against each baseline
    deck_pairs=[('artificer_tokens', 'broodmother_pack')],
)

# Evaluate built-in agents
print("Benchmark: Greedy Agent")
print("-" * 50)
greedy = GreedyAgent()
results = benchmark.evaluate(greedy)
print(results.summary())

In [ ]:
# Compare random agent
print("\nBenchmark: Random Agent")
print("-" * 50)
random_agent = RandomAgent()
results = benchmark.evaluate(random_agent)
print(results.summary())

## Deck Matchup Analysis

Analyze how different decks perform against each other.

In [ ]:
# Select some decks to analyze
decks_to_test = [
    'artificer_tokens',
    'broodmother_pack',
    'archon_burst',
    'grove_regenerate',
]

# Run matchups (using greedy vs greedy)
print("Deck Matchup Analysis (Greedy vs Greedy, 20 games each)")
print("=" * 60)
print(f"{'Deck 1':25s} vs {'Deck 2':25s} Win Rate")
print("-" * 60)

for d1 in decks_to_test:
    for d2 in decks_to_test:
        if d1 >= d2:  # Skip symmetric matchups
            continue
        
        result = evaluate_vs_bot(greedy_agent, 'greedy', num_games=20, deck1=d1, deck2=d2)
        print(f"{d1:25s} vs {d2:25s} {result['win_rate']*100:5.1f}%")

## Win Rate vs MCTS Strength

See how win rate changes as MCTS simulation count increases.

In [ ]:
import time

mcts_sims = [10, 25, 50, 100]  # MCTS simulation counts to test
num_games = 10  # Games per MCTS level (reduced for speed)

print("Greedy vs MCTS at various simulation counts")
print("-" * 50)

for sims in mcts_sims:
    start = time.time()
    wins = 0
    
    for seed in range(num_games):
        game = PyGame(deck1='artificer_tokens', deck2='broodmother_pack')
        game.reset(seed=seed)
        
        while not game.is_done():
            if game.current_player() == 0:
                action = game.greedy_action()
            else:
                action = game.mcts_action(sims)
            game.step(action)
        
        if game.winner() == 0:
            wins += 1
    
    elapsed = time.time() - start
    print(f"MCTS-{sims:3d}: {wins/num_games*100:5.1f}% greedy win rate ({elapsed:.1f}s)")

## Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Example: Plot win rates against different opponents
opponents = ['Random', 'Greedy', 'MCTS-50']
greedy_win_rates = [95, 50, 40]  # Example values

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(opponents, greedy_win_rates, color=['#2ecc71', '#3498db', '#e74c3c'])

# Add value labels on bars
for bar, val in zip(bars, greedy_win_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
            f'{val}%', ha='center', fontsize=12)

ax.axhline(50, color='gray', linestyle='--', alpha=0.5, label='50% baseline')
ax.set_ylabel('Win Rate (%)')
ax.set_title('Greedy Agent Performance vs Baselines')
ax.set_ylim(0, 100)
ax.legend()
plt.tight_layout()
plt.show()

## Next Steps

- **Train longer**: Increase training timesteps for better performance
- **Self-play**: Train against previous versions of your agent
- **AlphaZero**: Try `essence-wars train alphazero` for MCTS-guided training
- **CLI Benchmark**: Use `essence-wars benchmark --checkpoint model.pt` for full evaluation

For more information, see the [RESEARCHER_QUICKSTART.md](../docs/RESEARCHER_QUICKSTART.md).